# Visualizing Vibrational Modes

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook demonstrates how to compute and visualize vibrational modes for various molecules using the `symm4ml` library. We work through four examples of increasing complexity:

1. **Water ($C_{2v}$)** — 3 atoms, 3 vibrational modes
2. **Tetrahedron ($T_d$)** — 4 atoms, 6 vibrational modes
3. **Octahedron ($O_h$)** — 6 atoms, 9 vibrational modes
4. **Custom molecule with pymatgen** — template for your own molecules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/vib_modes_visualization.ipynb)

## Setup

In [ ]:
%%capture
!pip install pymatgen
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from symm4ml import groups, linalg, rep, vib_modes, plot
from symm4ml.utils import match_character_tables, print_character_table
from pymatgen.symmetry.groups import PointGroup
from pymatgen.core import Molecule
from pymatgen.symmetry.analyzer import PointGroupAnalyzer

## Plotting Functions

This notebook uses several plotting functions from `symm4ml.plot`:

**`plot.plot_projectors()`** — Heatmap of the projector equation $I - P^{\text{trans}} - P^{\text{rot}} = P^{\text{vib}}$ showing how degrees of freedom decompose.

**`plot.vibrational_modes_viewer()`** — Animated HTML viewer with mode cards showing atoms oscillating. Each mode gets a card with colored border and the irrep label. Includes projection selector buttons (xy/xz/yz) and depth cueing for 3D molecules.

We also use `match_character_tables()` from `symm4ml.utils` to align computed irreps with standard textbook labels.

---
## Example 1: Water ($C_{2v}$)

Water has 3 atoms and symmetry group $C_{2v}$ (4 elements, 4 irreps: $A_1, A_2, B_1, B_2$).

- Total DOF: $3 \times 3 = 9$
- Subtract translation (3) and rotation (3): **3 vibrational modes**
- Irrep decomposition: $\Gamma^{\text{vib}} = 2A_1 + B_1$

In [ ]:
# Water geometry: O at top, two H atoms below (matching course notes orientation)
# Bond angle ~104.5°, O-H distance ~0.96 Å
angle = np.radians(104.5)
water_vertices = np.array([
    [0.0, 0.0, 0.0],                                         # O
    [np.sin(angle/2), 0.0, -np.cos(angle/2)],                # H (below-right)
    [-np.sin(angle/2), 0.0, -np.cos(angle/2)],               # H (below-left)
])
# Center at origin
water_vertices -= water_vertices.mean(axis=0)

print("Water vertices (centered):")
print(np.round(water_vertices, 3))

In [ ]:
# Get C2v symmetry operations and find irreps
C2v_vec = vib_modes.C2v_vec
C2v_table = vib_modes.C2v_table

np.random.seed(42)
C2v_irreps = rep.infer_irreps(C2v_table)
C2v_conj_classes = groups.conjugacy_classes(C2v_table)
C2v_char_table = rep.character_table(C2v_irreps, C2v_conj_classes)

# Reference character table for C2v (Dresselhaus convention)
# Columns: E, C₂, σᵥ(xz), σᵥ'(yz)
C2v_irrep_names = ['A₁', 'A₂', 'B₁', 'B₂']
C2v_class_names = ['E', 'C₂', 'σᵥ(xz)', "σᵥ'(yz)"]
C2v_ref_chars = np.array([
    [ 1,  1,  1,  1],   # A₁
    [ 1,  1, -1, -1],   # A₂
    [ 1, -1,  1, -1],   # B₁
    [ 1, -1, -1,  1],   # B₂
])

# Match computed table to reference
row_perm, col_perm = match_character_tables(C2v_char_table, C2v_ref_chars)
C2v_irreps = [C2v_irreps[i] for i in row_perm]
C2v_conj_classes_list = [list(C2v_conj_classes)[j] for j in col_perm]
C2v_char_table = rep.character_table(C2v_irreps, C2v_conj_classes_list)

print_character_table(C2v_char_table, C2v_irrep_names, C2v_class_names,
                      title="Character table of C₂ᵥ")
print(f"\nIrrep dimensions: {[ir.shape[1] for ir in C2v_irreps]}")

In [ ]:
# Compute vibrational modes (using matched irreps)
water_modes = vib_modes.vibrational_modes(water_vertices, C2v_vec, C2v_irreps)

# Show which irreps have vibrational modes
for i, m in enumerate(water_modes):
    if m.shape[0] > 0:
        print(f"{C2v_irrep_names[i]} (dim={C2v_irreps[i].shape[1]}): {m.shape[0]} mode(s)")

# Decomposition formula for chi^vib
C2v_class_sizes = [len(c) for c in C2v_conj_classes_list]
chi_vib = np.array([m.shape[0] * C2v_irreps[i].shape[1] for i, m in enumerate(water_modes)])
decomp_str = ' + '.join(
    f"{m.shape[0]}{C2v_irrep_names[i]}" for i, m in enumerate(water_modes) if m.shape[0] > 0
)
print(f"\nΓ_vib = {decomp_str}")

In [ ]:
# Projector decomposition for water
fig = plot.plot_projectors(water_vertices)
plt.show()

In [ ]:
# Animated vibrational mode viewer
HTML(plot.vibrational_modes_viewer(
    water_vertices,
    water_modes,
    irrep_labels=C2v_irrep_names,
    bonds=[[0, 1], [0, 2]],
    atom_colors=["#e53935", "#1565c0", "#00897b"],  # O=red, H1=blue, H2=teal
    atom_sizes=[16, 12, 12],
    atom_labels=["O", "H", "H"],
    scale=0.3,
))

**Discussion**: Water has $2A_1 + B_1$ vibrational modes:
- The two $A_1$ modes are the **symmetric stretch** and **symmetric bend**. By Schur's lemma, the Hamiltonian can mix modes within the same irrep — symmetry alone doesn't distinguish stretch from bend.
- The single $B_1$ mode is the **asymmetric stretch** — it cannot mix with the $A_1$ modes.

---
## Example 2: Tetrahedron ($T_d$)

4 identical atoms at the corners of a regular tetrahedron, with symmetry group $T_d$ (24 elements, 5 irreps: $A_1, A_2, E, T_1, T_2$).

- Total DOF: $3 \times 4 = 12$
- Subtract translation (3) and rotation (3): **6 vibrational modes**
- Irrep decomposition: $\Gamma^{\text{vib}} = A_1 + E + T_2$

In [ ]:
# Tetrahedron vertices (already centered) and Td symmetry data
tetrahedron = vib_modes.tetrahedron
Td_vec = vib_modes.Td_vec
Td_table = vib_modes.Td_table
Td_irreps = list(vib_modes.Td_irreps)  # mutable copy

# Build character table and match to reference
Td_conj_classes = groups.conjugacy_classes(Td_table)
Td_char_table = rep.character_table(Td_irreps, Td_conj_classes)

# Reference character table for Td (Dresselhaus convention)
# Columns: E, 8C₃, 3C₂, 6S₄, 6σ_d
Td_irrep_names = ['A₁', 'A₂', 'E', 'T₁', 'T₂']
Td_class_names = ['E', '8C₃', '3C₂', '6S₄', '6σ_d']
Td_ref_chars = np.array([
    [ 1,  1,  1,  1,  1],   # A₁
    [ 1,  1,  1, -1, -1],   # A₂
    [ 2, -1,  2,  0,  0],   # E
    [ 3,  0, -1,  1, -1],   # T₁
    [ 3,  0, -1, -1,  1],   # T₂
])

row_perm, col_perm = match_character_tables(Td_char_table, Td_ref_chars)
Td_irreps = [Td_irreps[i] for i in row_perm]
Td_conj_classes_list = [list(Td_conj_classes)[j] for j in col_perm]
Td_char_table = rep.character_table(Td_irreps, Td_conj_classes_list)

print_character_table(Td_char_table, Td_irrep_names, Td_class_names,
                      title="Character table of T_d")
print(f"\nIrrep dimensions: {[ir.shape[1] for ir in Td_irreps]}")
print(f"Tetrahedron vertices:\n{tetrahedron}")

In [ ]:
# Compute vibrational modes (using matched irreps)
tetra_modes = vib_modes.vibrational_modes(tetrahedron, Td_vec, Td_irreps)

for i, m in enumerate(tetra_modes):
    if m.shape[0] > 0:
        print(f"{Td_irrep_names[i]} (dim={Td_irreps[i].shape[1]}): {m.shape[0]} mode(s)")

decomp_str = ' + '.join(
    f"{m.shape[0]}{Td_irrep_names[i]}" if m.shape[0] > 1 else Td_irrep_names[i]
    for i, m in enumerate(tetra_modes) if m.shape[0] > 0
)
print(f"\nΓ_vib = {decomp_str}")

In [ ]:
# Projector decomposition for the tetrahedron
fig = plot.plot_projectors(tetrahedron)
plt.show()

In [ ]:
# Animated vibrational mode viewer
tetra_bonds = [[i, j] for i in range(4) for j in range(i+1, 4)]

HTML(plot.vibrational_modes_viewer(
    tetrahedron,
    tetra_modes,
    irrep_labels=Td_irrep_names,
    bonds=tetra_bonds,
    scale=0.4,
))

---
## Example 3: Octahedron ($O_h$)

6 identical atoms at the corners of a regular octahedron, with symmetry group $O_h$ (48 elements, 10 irreps).

- Total DOF: $3 \times 6 = 18$
- Subtract translation (3) and rotation (3): **12 vibrational modes** (but some may be degenerate, giving fewer distinct frequencies)

In [ ]:
# Octahedron vertices (already centered) and Oh symmetry data
octahedron = vib_modes.octahedron
Oh_vec = vib_modes.Oh_vec
Oh_table = vib_modes.Oh_table

np.random.seed(42)
Oh_irreps = rep.infer_irreps(Oh_table)
Oh_conj_classes = groups.conjugacy_classes(Oh_table)
Oh_char_table = rep.character_table(Oh_irreps, Oh_conj_classes)

# Reference character table for Oh (Dresselhaus convention)
# Columns: E, 8C₃, 6C₂, 6C₄, 3C₂(=C₄²), i, 6S₄, 8S₆, 3σ_h, 6σ_d
Oh_irrep_names = ['A₁g', 'A₂g', 'Eg', 'T₁g', 'T₂g',
                  'A₁u', 'A₂u', 'Eu', 'T₁u', 'T₂u']
Oh_class_names = ['E', '8C₃', '6C₂', '6C₄', '3C₂\'',
                  'i', '6S₄', '8S₆', '3σ_h', '6σ_d']
Oh_ref_chars = np.array([
    [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],   # A₁g
    [ 1,  1, -1, -1,  1,  1, -1,  1,  1, -1],   # A₂g
    [ 2, -1,  0,  0,  2,  2,  0, -1,  2,  0],   # Eg
    [ 3,  0, -1,  1, -1,  3,  1,  0, -1, -1],   # T₁g
    [ 3,  0,  1, -1, -1,  3, -1,  0, -1,  1],   # T₂g
    [ 1,  1,  1,  1,  1, -1, -1, -1, -1, -1],   # A₁u
    [ 1,  1, -1, -1,  1, -1,  1, -1, -1,  1],   # A₂u
    [ 2, -1,  0,  0,  2, -2,  0,  1, -2,  0],   # Eu
    [ 3,  0, -1,  1, -1, -3, -1,  0,  1,  1],   # T₁u
    [ 3,  0,  1, -1, -1, -3,  1,  0,  1, -1],   # T₂u
])

row_perm, col_perm = match_character_tables(Oh_char_table, Oh_ref_chars)
Oh_irreps = [Oh_irreps[i] for i in row_perm]
Oh_conj_classes_list = [list(Oh_conj_classes)[j] for j in col_perm]
Oh_char_table = rep.character_table(Oh_irreps, Oh_conj_classes_list)

print_character_table(Oh_char_table, Oh_irrep_names, Oh_class_names,
                      title="Character table of O_h")
print(f"\nIrrep dimensions: {[ir.shape[1] for ir in Oh_irreps]}")

In [ ]:
# Compute vibrational modes (using matched irreps)
oct_modes = vib_modes.vibrational_modes(octahedron, Oh_vec, Oh_irreps)

total_modes = 0
for i, m in enumerate(oct_modes):
    if m.shape[0] > 0:
        print(f"{Oh_irrep_names[i]} (dim={Oh_irreps[i].shape[1]}): {m.shape[0]} mode(s)")
        total_modes += m.shape[0]

decomp_str = ' + '.join(
    f"{m.shape[0]}{Oh_irrep_names[i]}" if m.shape[0] > 1 else Oh_irrep_names[i]
    for i, m in enumerate(oct_modes) if m.shape[0] > 0
)
print(f"\nΓ_vib = {decomp_str}")
print(f"Total vibrational modes: {total_modes} (expected: 3×6 - 6 = 12)")

In [ ]:
# Animated vibrational mode viewer
oct_bonds = []
for i in range(6):
    for j in range(i+1, 6):
        if np.linalg.norm(octahedron[i] - octahedron[j]) < 1.5:
            oct_bonds.append([i, j])

HTML(plot.vibrational_modes_viewer(
    octahedron,
    oct_modes,
    irrep_labels=Oh_irrep_names,
    bonds=oct_bonds,
    scale=0.5, canvas_size=160, atom_sizes=6*[7]
))

**Note**: The irreps are labeled using the standard $O_h$ notation. The subscript $g$ (gerade) means even under inversion; $u$ (ungerade) means odd. The vibrational modes live in the $A_{1g}$, $E_g$, and $T_{1u}$ irreps — consistent with the known result for octahedral molecules.

---
## Example 4: Custom Molecule with Pymatgen

Here's a template for analyzing any molecule. You need:
1. Atom positions (centered at the origin)
2. The atom species (element symbols)

We use pymatgen's `PointGroupAnalyzer` to identify the symmetry group $G$ and obtain the vector representation with operations matching the orientation of the molecule.

We'll use **ammonia ($\text{NH}_3$)** with symmetry group $C_{3v}$ as an example.

In [ ]:
# Step 1: Define molecule geometry (centered at origin)
# Ammonia: N at top, 3 H atoms forming a triangle below
nh_dist = 1.012  # N-H bond length in Angstroms

# Place N at origin, H atoms below
theta = np.arccos(-1/3)  # angle from z-axis to N-H bond
r = nh_dist * np.sin(theta)  # radial distance of H from z-axis
z_h = nh_dist * np.cos(theta)  # z-coordinate of H atoms

ammonia = np.array([
    [0.0, 0.0, 0.0],                                                      # N
    [r * np.cos(np.pi/2), r * np.sin(np.pi/2), z_h],                      # H1
    [r * np.cos(np.pi/2 + 2*np.pi/3), r * np.sin(np.pi/2 + 2*np.pi/3), z_h],   # H2
    [r * np.cos(np.pi/2 + 4*np.pi/3), r * np.sin(np.pi/2 + 4*np.pi/3), z_h],   # H3
])

# Center at origin
ammonia -= ammonia.mean(axis=0)
print("Ammonia vertices (centered):")
print(np.round(ammonia, 3))

In [ ]:
# Step 2: Get point group symmetry operations from the molecule geometry
# Using PointGroupAnalyzer ensures the symmetry operations match the
# actual orientation of the molecule (no manual alignment needed)
species = ["N", "H", "H", "H"]
mol = Molecule(species, ammonia)
pga = PointGroupAnalyzer(mol)

print(f"Detected point group: {pga.get_pointgroup()}")

# Extract the 3x3 rotation matrices as the vector representation
symmops = pga.get_symmetry_operations()
vec_rep = np.stack([op.rotation_matrix for op in symmops], axis=0)

print(f"Number of symmetry operations: {len(vec_rep)}")

In [ ]:
# Step 3: Build multiplication table, find irreps, and match to reference
table = groups.make_multiplication_table(vec_rep)

np.random.seed(42)
irreps = rep.infer_irreps(table)
conj_classes = groups.conjugacy_classes(table)
char_table = rep.character_table(irreps, conj_classes)

# Reference character table for C3v (Dresselhaus convention)
# Columns: E, 2C₃, 3σᵥ
C3v_irrep_names = ['A₁', 'A₂', 'E']
C3v_class_names = ['E', '2C₃', '3σᵥ']
C3v_ref_chars = np.array([
    [1,  1,  1],   # A₁
    [1,  1, -1],   # A₂
    [2, -1,  0],   # E
])

row_perm, col_perm = match_character_tables(char_table, C3v_ref_chars)
irreps = [irreps[i] for i in row_perm]
conj_classes_list = [list(conj_classes)[j] for j in col_perm]
char_table = rep.character_table(irreps, conj_classes_list)

print_character_table(char_table, C3v_irrep_names, C3v_class_names,
                      title="Character table of C₃ᵥ")
print(f"\nIrrep dimensions: {[ir.shape[1] for ir in irreps]}")

In [ ]:
# Step 4: Compute vibrational modes (using matched irreps)
modes = vib_modes.vibrational_modes(ammonia, vec_rep, irreps)

total = 0
for i, m in enumerate(modes):
    if m.shape[0] > 0:
        print(f"{C3v_irrep_names[i]} (dim={irreps[i].shape[1]}): {m.shape[0]} mode(s)")
        total += m.shape[0]

decomp_str = ' + '.join(
    f"{m.shape[0]}{C3v_irrep_names[i]}" if m.shape[0] > 1 else C3v_irrep_names[i]
    for i, m in enumerate(modes) if m.shape[0] > 0
)
n_atoms = len(ammonia)
expected = 3 * n_atoms - 6
print(f"\nΓ_vib = {decomp_str}")
print(f"Total: {total} modes (expected: 3×{n_atoms} - 6 = {expected})")

In [ ]:
# Step 5: Visualize
ammonia_bonds = [[0, 1], [0, 2], [0, 3]]  # N-H bonds

fig = plot.plot_projectors(ammonia)
plt.show()

In [ ]:
# Animated vibrational mode viewer
HTML(plot.vibrational_modes_viewer(
    ammonia,
    modes,
    irrep_labels=C3v_irrep_names,
    bonds=ammonia_bonds,
    atom_colors=["#1565C0", "#e53935", "#00897b", "#e65100"],  # N=blue, H1=red, H2=teal, H3=orange
    atom_sizes=[16, 12, 12, 12],
    atom_labels=["N", "H", "H", "H"],
    scale=0.3,
))

---
## Template: Analyze Your Own Molecule

Copy the cells below and modify them for your molecule of interest.

```python
from pymatgen.core import Molecule
from pymatgen.symmetry.analyzer import PointGroupAnalyzer

# 1. Define molecule (centered at origin)
vertices = np.array([...])  # (n, 3)
vertices -= vertices.mean(axis=0)  # center

# 2. Identify symmetry group and get vector representation
#    PointGroupAnalyzer determines the operations from the actual
#    molecular geometry, so they match the molecule's orientation.
species = ["O", "H", "H"]  # your atom species
mol = Molecule(species, vertices)
pga = PointGroupAnalyzer(mol)
print(f"Detected point group: {pga.get_pointgroup()}")

symmops = pga.get_symmetry_operations()
vec_rep = np.stack([op.rotation_matrix for op in symmops], axis=0)

# 3. Build multiplication table, find irreps, and match to reference
table = groups.make_multiplication_table(vec_rep)
np.random.seed(42)
irreps = rep.infer_irreps(table)
conj_classes = groups.conjugacy_classes(table)
char_table = rep.character_table(irreps, conj_classes)

# Define your reference character table (from textbook)
irrep_names = ['A₁', 'A₂', 'B₁', 'B₂']  # your irrep names
ref_chars = np.array([...])  # your reference character table
row_perm, col_perm = match_character_tables(char_table, ref_chars)
irreps = [irreps[i] for i in row_perm]

# 4. Compute vibrational modes
modes = vib_modes.vibrational_modes(vertices, vec_rep, irreps)

# 5. Visualize
fig = plot.plot_projectors(vertices)
plt.show()

HTML(plot.vibrational_modes_viewer(
    vertices, modes,
    irrep_labels=irrep_names,
    bonds=[[0,1], [0,2]],
))
```

### Available point groups

The `vib_modes.point_group_dict` maps Schoenflies symbols to Hermann-Mauguin notation used by pymatgen:

In [ ]:
for schoenflies, hm in sorted(vib_modes.point_group_dict.items()):
    print(f"  {schoenflies:6s} -> {hm}")